# Tokio & Async Rust - Complete Beginner Guide

**What is this notebook?**

This is a hands-on, interactive guide that teaches you **async programming in Rust** using **Tokio** from absolute zero. Every cell builds on the previous one. Read the theory, then run the code, and see what happens.

---

## Before We Start: Setup

This notebook uses the **evcxr** Rust kernel for Jupyter. To install it:

```bash
# Step 1: Install Jupyter if you don't have it
pip install jupyter

# Step 2: Install the Rust Jupyter kernel
cargo install evcxr_jupyter
evcxr_jupyter --install

# Step 3: Open this notebook
jupyter notebook tokio_and_async_guide.ipynb
```

If you can't set up Jupyter, **no worries!** Each code cell is also a standalone example. You can copy any cell's code into a `.rs` file and run it with `cargo run`.

---

# Chapter 1: What is Tokio and Why Are We Learning It?

---

## The Simple Answer

**Tokio** is a tool (a library) for Rust that lets your program **do many things at the same time** without wasting time waiting around.

## The Real-World Analogy

Imagine you're making breakfast:

**The slow way (synchronous/blocking):**
1. Put bread in toaster. Stand there. Stare at toaster. Wait 3 minutes.
2. Toast is done. Now boil water for tea. Stand there. Wait 4 minutes.
3. Water is done. Now fry an egg. Stand there. Wait 2 minutes.
4. **Total time: 9 minutes** (3 + 4 + 2)

**The smart way (asynchronous):**
1. Put bread in toaster.
2. While toast is cooking, start boiling water.
3. While both are going, start frying the egg.
4. Toast pops up - grab it. Egg is done - plate it. Water boils - make tea.
5. **Total time: ~4 minutes** (everything happened at the same time!)

You are **one person** (one CPU thread), but you handled **three tasks** by not wasting time standing around.

**That's what Tokio does for your Rust program.**

## Why Do We Need This?

Programs often wait for things:
- Waiting for a website to respond to your request
- Waiting for a file to be read from disk
- Waiting for a user to connect to your server
- Waiting for a database query to finish

During all this waiting, the CPU is **sitting idle doing nothing**. If you have 10,000 users connecting to your server, and each connection involves waiting, you're wasting enormous amounts of time.

Tokio solves this. It says: "While you're waiting for User 1's data, go handle User 2. When User 1's data arrives, come back to them."

## Why Tokio Specifically?

Rust has `async` and `.await` keywords built into the language, but it does **not** have a built-in engine to actually run async code. You need to pick an engine (called a **runtime**). Tokio is:

- The **most popular** async runtime in Rust
- Used by huge companies in production
- Extremely fast and reliable
- Has great tools for networking, timers, file I/O, etc.

Other runtimes exist (`async-std`, `smol`), but Tokio is the standard.

## Why Are YOU Learning It?

You're building toward a **reverse proxy** project. A reverse proxy handles many network connections at the same time — accepting requests, forwarding them, waiting for responses, sending them back. Without async, this would be impossibly slow or require thousands of threads. Tokio makes it practical.

Your TL Takashi's advice: understand Tokio deeply first, and the reverse proxy code will make sense naturally.

---

# Chapter 2: Synchronous vs Asynchronous — See The Difference

Let's start with actual code. In this first cell, we'll see what **synchronous (blocking)** code looks like, and why it's a problem.

### Cell 1: The Blocking Problem (Conceptual)

Read this code. **Don't run it yet** — just understand what it does.

```rust
use std::thread;
use std::time::Duration;

fn make_toast() -> String {
    println!("[Toast] Putting bread in toaster...");
    thread::sleep(Duration::from_secs(3));  // Blocks! CPU does nothing for 3 seconds
    println!("[Toast] Done!");
    "toast".to_string()
}

fn boil_water() -> String {
    println!("[Water] Turning on kettle...");
    thread::sleep(Duration::from_secs(4));  // Blocks! CPU does nothing for 4 seconds
    println!("[Water] Done!");
    "hot water".to_string()
}

fn fry_egg() -> String {
    println!("[Egg] Cracking egg into pan...");
    thread::sleep(Duration::from_secs(2));  // Blocks! CPU does nothing for 2 seconds
    println!("[Egg] Done!");
    "fried egg".to_string()
}

fn main() {
    let start = std::time::Instant::now();
    
    let toast = make_toast();     // Wait 3 seconds...
    let water = boil_water();     // THEN wait 4 seconds...
    let egg = fry_egg();          // THEN wait 2 seconds...
    
    println!("\nBreakfast ready: {}, {}, {}", toast, water, egg);
    println!("Total time: {:?}", start.elapsed());  // ~9 seconds!
}
```

**What's happening:** Each function blocks (freezes) the program while it "works." `make_toast` takes 3 seconds, THEN `boil_water` takes 4, THEN `fry_egg` takes 2. Total: **9 seconds**.

This is dumb. You wouldn't cook breakfast this way. Let's fix it with async.

### Setting Up Tokio in This Notebook

In the next cell, we tell the Jupyter kernel to load the Tokio library. Think of this like adding `tokio` to your `Cargo.toml` file. **Run this cell first before any other code cell.**

In [ ]:
// SETUP CELL - Run this first!
// This tells evcxr to download and use Tokio.
// It's the same as adding tokio to Cargo.toml.

:dep tokio = { version = "1", features = ["full"] }

If the cell above ran without errors, Tokio is loaded and ready. If you see an error, make sure you installed `evcxr_jupyter` correctly.

---

### Cell 2: Your Very First Async Code

Now let's do the same breakfast, but **asynchronously** with Tokio.

**What to watch for in this cell:**
- `async fn` — means "this function can pause and let others run"
- `tokio::time::sleep` — the async version of sleeping (doesn't block!)
- `.await` — means "pause here, let other tasks run, come back when ready"
- `tokio::join!` — means "run all these at the same time, wait for all to finish"

In [ ]:
use tokio::time::{sleep, Duration, Instant};

async fn make_toast() -> String {
    println!("[Toast] Putting bread in toaster...");
    sleep(Duration::from_secs(3)).await;  // Non-blocking! Other tasks can run!
    println!("[Toast] Done!");
    "toast".to_string()
}

async fn boil_water() -> String {
    println!("[Water] Turning on kettle...");
    sleep(Duration::from_secs(4)).await;  // Non-blocking!
    println!("[Water] Done!");
    "hot water".to_string()
}

async fn fry_egg() -> String {
    println!("[Egg] Cracking egg into pan...");
    sleep(Duration::from_secs(2)).await;  // Non-blocking!
    println!("[Egg] Done!");
    "fried egg".to_string()
}

// In a notebook, we use tokio::runtime to run async code
let rt = tokio::runtime::Runtime::new().unwrap();
rt.block_on(async {
    let start = Instant::now();
    
    // Run ALL THREE at the same time!
    let (toast, water, egg) = tokio::join!(
        make_toast(),
        boil_water(),
        fry_egg()
    );
    
    println!("\nBreakfast ready: {}, {}, {}", toast, water, egg);
    println!("Total time: {:?}", start.elapsed());  // ~4 seconds! (not 9)
});

### What Just Happened?

**Look at the output.** You should see:
1. All three tasks **started immediately** (all three "starting" messages appear)
2. The **egg finished first** (only 2 seconds)
3. The **toast finished next** (3 seconds)
4. The **water finished last** (4 seconds)
5. Total time was **~4 seconds**, not 9!

**Why?** Because `tokio::join!` ran all three functions at the same time. When `make_toast` hit `.await` on its sleep, Tokio said "okay, you're waiting, let me check on the others." It kept switching between tasks, and all three made progress simultaneously.

**This is the core idea of async: don't waste time waiting. Do other things while you wait.**

---

# Chapter 3: Understanding `async fn` and Futures

---

## What Does `async fn` Actually Mean?

When you write a normal function:
```rust
fn add(a: i32, b: i32) -> i32 {
    a + b
}
```
You call it, it runs, it gives you the answer. Done.

When you write an `async fn`:
```rust
async fn add(a: i32, b: i32) -> i32 {
    a + b
}
```
You call it and it gives you... **a Future**. NOT the answer. A Future is like a **promise**: "I will give you an `i32` eventually, but not right now."

Think of it like this:
- **Normal function** = you ask someone a question, they answer immediately
- **Async function** = you ask someone a question, they hand you a buzzer and say "I'll buzz you when I have the answer"

The `.await` is you pressing the buzzer and saying "I'm ready to wait for the answer now."

Let's prove this with code.

### Cell 3: Calling an Async Function Without `.await`

**What to watch for:** When you call an async function without `.await`, it does **nothing**. It just creates a Future that sits there. The code inside the function **does not run**.

In [ ]:
async fn say_hello() {
    println!("Hello! This function actually ran!");
}

let rt = tokio::runtime::Runtime::new().unwrap();
rt.block_on(async {
    
    // Calling WITHOUT .await
    println!("--- Calling say_hello() WITHOUT await ---");
    let _future = say_hello();  // This creates a Future but does NOT run the function!
    println!("Did you see 'Hello'? No! The function didn't run.\n");
    
    // Calling WITH .await
    println!("--- Calling say_hello() WITH await ---");
    say_hello().await;  // NOW it actually runs!
    println!("Now you saw 'Hello'! The function ran because we used .await");
});

### What Just Happened?

The first call `say_hello()` (without `.await`) created a Future but **never ran the function body**. You didn't see "Hello! This function actually ran!" printed.

The second call `say_hello().await` actually **executed** the function and you saw the print.

**Key lesson: An `async fn` is lazy. It only runs when you `.await` it (or when a runtime polls it).**

This is different from most other languages! In JavaScript, calling an async function starts it immediately. In Rust, calling an async function does **nothing** until you await it.

---

### Cell 4: Futures Return Values

**What to watch for:** An `async fn` that says it returns `-> String` doesn't directly return a String. It returns a Future that **eventually produces** a String. You get the actual String only when you `.await`.

In [ ]:
async fn get_username() -> String {
    // Imagine this goes to a database (we simulate with sleep)
    sleep(Duration::from_millis(500)).await;
    "Sujit".to_string()
}

async fn get_age() -> u32 {
    // Imagine this goes to a different database
    sleep(Duration::from_millis(300)).await;
    25
}

let rt = tokio::runtime::Runtime::new().unwrap();
rt.block_on(async {
    
    // Get both values - they run one after the other (sequentially)
    println!("--- Sequential (one after another) ---");
    let start = Instant::now();
    let name = get_username().await;  // Wait 500ms
    let age = get_age().await;        // Then wait 300ms
    println!("User: {}, Age: {}", name, age);
    println!("Time: {:?}\n", start.elapsed());  // ~800ms (500 + 300)
    
    // Get both values - they run at the same time (concurrently)
    println!("--- Concurrent (at the same time) ---");
    let start = Instant::now();
    let (name, age) = tokio::join!(get_username(), get_age());  // Both at once!
    println!("User: {}, Age: {}", name, age);
    println!("Time: {:?}", start.elapsed());  // ~500ms (the slower one)
});

### What Just Happened?

**Sequential version:** We awaited `get_username()` first (500ms), then `get_age()` (300ms). Total: **~800ms**.

**Concurrent version:** We used `tokio::join!` to run both at the same time. Total: **~500ms** (only as slow as the slowest one).

**When to use which:**
- Use sequential `.await` when the second task **depends on** the first (e.g., you need the username before you can look up their age)
- Use `tokio::join!` when the tasks are **independent** (e.g., username and age come from different sources)

---

# Chapter 4: The Tokio Runtime — The Engine Behind Everything

---

## What Is a Runtime?

Rust's `async` and `.await` are like writing a **recipe**. But a recipe doesn't cook itself — you need a **kitchen** with an oven, stove, and a cook. 

**The Tokio Runtime is that kitchen.**

It has three main parts:

### 1. The Executor (The Cook)
The executor takes your async tasks and actually runs them. It keeps a list of all tasks, and when one pauses at an `.await`, it picks up another task and works on that.

### 2. The Reactor (The Timer/Bell System)
The reactor watches for external events — "has data arrived on this network connection?", "has this timer expired?". When something happens, it tells the executor: "Hey, that task can make progress now!"

### 3. The Task Queue (The Order Board)
All the tasks that are ready to be worked on sit in a queue. The executor picks from this queue.

### How They Work Together:
```
1. You spawn a task (put an order on the board)
2. Executor picks it up and starts running it
3. Task hits .await (waiting for network data)
4. Executor puts it aside, picks up another task
5. Reactor notices: "network data arrived!"
6. Reactor puts the paused task back on the queue
7. Executor picks it up again and continues from where it paused
8. Task finishes -> result is ready
```

## Two Types of Runtime

Tokio gives you two options:

**1. Multi-threaded runtime** (`#[tokio::main]` or `Runtime::new()`)
- Uses multiple CPU threads (default: one per CPU core)
- Tasks can move between threads
- Best for servers and heavy workloads

**2. Single-threaded runtime** (`#[tokio::main(flavor = "current_thread")]`)
- Uses only one thread
- Simpler, less overhead
- Good for simple programs or when you don't need parallelism

### Cell 5: Seeing the Runtime in Action

**What to watch for:** We'll create both types of runtimes and see which threads our tasks run on. Notice how the multi-threaded runtime uses different threads.

In [ ]:
use std::thread;

async fn which_thread(task_name: &str) {
    println!(
        "  {} is running on thread: {:?}",
        task_name,
        thread::current().name().unwrap_or("unnamed")
    );
    // Yield to let other tasks run
    tokio::task::yield_now().await;
    println!(
        "  {} resumed on thread: {:?}",
        task_name,
        thread::current().name().unwrap_or("unnamed")
    );
}

// === Single-threaded runtime ===
println!("=== Single-threaded runtime ===");
let rt = tokio::runtime::Builder::new_current_thread()
    .enable_all()
    .build()
    .unwrap();
rt.block_on(async {
    // Both tasks run on the same thread
    tokio::join!(which_thread("Task A"), which_thread("Task B"));
});

println!();

// === Multi-threaded runtime ===
println!("=== Multi-threaded runtime ===");
let rt = tokio::runtime::Builder::new_multi_thread()
    .worker_threads(2)  // Use 2 threads so we can see the difference
    .enable_all()
    .build()
    .unwrap();
rt.block_on(async {
    // Tasks might run on different threads!
    let a = tokio::spawn(which_thread("Task A"));
    let b = tokio::spawn(which_thread("Task B"));
    a.await.unwrap();
    b.await.unwrap();
});

### What Just Happened?

- **Single-threaded:** Both Task A and Task B ran on the **same thread**. They took turns, but never ran truly in parallel.
- **Multi-threaded:** Task A and Task B might have run on **different threads**. They could run truly in parallel on separate CPU cores.

**But here's the important thing:** Even in the single-threaded runtime, both tasks made progress "at the same time" (concurrency). The single thread switched between them at every `.await` point. Concurrency (taking turns quickly) is not the same as parallelism (actually running at the exact same instant).

---

# Chapter 5: `tokio::spawn` — Creating Independent Tasks

---

## What is `tokio::spawn`?

So far we've been using `tokio::join!` to run things at the same time. But `join!` makes you wait for ALL of them to finish before moving on.

**`tokio::spawn`** is different. It starts a task **in the background** and gives you a handle. The task runs on its own, and you can check on it later (or never).

Think of it like this:
- `join!` = "Do these 3 things. I'll wait here until ALL 3 are done."
- `spawn` = "Go do this in the background. I'll keep doing my own stuff. I'll check on you later."

### The `'static` Rule

There's one important rule: **spawned tasks must own all their data**. You can't borrow data from outside because the task might outlive the data.

```rust
// This WON'T compile:
let name = String::from("Sujit");
tokio::spawn(async {
    println!("{}", name);  // ERROR: `name` is borrowed, not owned
});

// This WILL compile:
let name = String::from("Sujit");
tokio::spawn(async move {  // `move` transfers ownership
    println!("{}", name);  // OK: the task now owns `name`
});
// `name` is gone here - the task took it
```

### Cell 6: Spawning Background Tasks

**What to watch for:** We spawn 3 tasks that each "download" something (simulated with sleep). While they run in the background, the main task keeps doing its own work. Watch the **order** of the printed messages!

In [ ]:
let rt = tokio::runtime::Runtime::new().unwrap();
rt.block_on(async {
    let start = Instant::now();
    
    // Spawn 3 background "download" tasks
    let task1 = tokio::spawn(async {
        println!("  [{:?}] Download A: starting...", Instant::now());
        sleep(Duration::from_secs(3)).await;
        println!("  [{:?}] Download A: finished!", Instant::now());
        1000  // return the "file size"
    });
    
    let task2 = tokio::spawn(async {
        println!("  [{:?}] Download B: starting...", Instant::now());
        sleep(Duration::from_secs(1)).await;
        println!("  [{:?}] Download B: finished!", Instant::now());
        500
    });
    
    let task3 = tokio::spawn(async {
        println!("  [{:?}] Download C: starting...", Instant::now());
        sleep(Duration::from_secs(2)).await;
        println!("  [{:?}] Download C: finished!", Instant::now());
        750
    });
    
    // While downloads run in the background, we can do other work!
    println!("\n  Main task: All downloads started! Doing other work...");
    sleep(Duration::from_millis(500)).await;
    println!("  Main task: Still working on my own stuff...\n");
    
    // Now let's collect the results
    let size1 = task1.await.unwrap();
    let size2 = task2.await.unwrap();
    let size3 = task3.await.unwrap();
    
    println!("\n  All downloads complete!");
    println!("  Total bytes: {}", size1 + size2 + size3);
    println!("  Total time: {:?}", start.elapsed());  // ~3 seconds, not 6!
});

### What Just Happened?

1. We **spawned** 3 download tasks — they all started running immediately in the background
2. The main task kept going and did its own work ("Doing other work...")
3. Downloads finished in order of their duration: B (1s), C (2s), A (3s)
4. We collected all results with `.await` on each handle
5. Total time: **~3 seconds** (the slowest download), not 6 seconds (1+2+3)

**Key difference from `join!`:**
- `join!` starts tasks and **blocks the current task** until all are done
- `spawn` starts tasks **in the background** and lets you keep working; you collect results whenever you want

---

# Chapter 6: How Futures REALLY Work Inside (The Polling Model)

---

This is where we go deeper — this is what your TL wants you to understand.

## The `Future` Trait

Behind the scenes, every async function becomes a struct that implements the `Future` trait:

```rust
use std::future::Future;
use std::pin::Pin;
use std::task::{Context, Poll};

trait Future {
    type Output;  // What type of value this future produces
    
    fn poll(self: Pin<&mut Self>, cx: &mut Context<'_>) -> Poll<Self::Output>;
}

enum Poll<T> {
    Ready(T),   // "Done! Here's the result."
    Pending,    // "Not ready yet. I'll tell you when I am."
}
```

**How Tokio uses this:**
1. Tokio calls `poll()` on your Future
2. If it returns `Ready(value)` — great, the Future is done
3. If it returns `Pending` — the Future is waiting for something
4. Before returning `Pending`, the Future saves a **Waker** (from the Context)
5. When the thing it's waiting for is ready, the Waker is called
6. The Waker tells Tokio: "poll this Future again!"
7. Tokio polls again, and this time it returns `Ready(value)`

## What is a Waker?

A Waker is an alarm bell. When a Future returns `Pending`, it's saying:
> "I'm not done. But I've set an alarm. When the alarm goes off, please poll me again."

For example, if a Future is waiting for network data:
- It tells the OS: "notify me when data arrives on this socket"
- It saves the Waker
- It returns `Pending`
- Data arrives → OS notifies → Waker fires → Tokio polls again → Future returns `Ready(data)`

**Tokio never busy-loops checking "are you done yet? are you done yet?"** — it only polls when the Waker says to.

### Cell 7: Build Your Own Future From Scratch

**What to watch for:** We're going to build a **custom Future** by hand — no `async fn`, no `.await`. Just implementing the `Future` trait directly. This is what the compiler does behind the scenes when you write `async fn`.

We'll make a countdown timer that counts down from 3 to 0.

In [ ]:
use std::future::Future;
use std::pin::Pin;
use std::task::{Context, Poll};

// Our custom Future: counts down from a number to zero
struct Countdown {
    count: u32,
}

impl Future for Countdown {
    type Output = String;  // When done, it produces a String
    
    fn poll(mut self: Pin<&mut Self>, cx: &mut Context<'_>) -> Poll<String> {
        if self.count == 0 {
            // We're done! Return the final result.
            println!("    poll() called -> Ready! (Liftoff!)");
            Poll::Ready("Liftoff!".to_string())
        } else {
            // Not done yet. Print current count and decrease.
            println!("    poll() called -> Pending (count = {})", self.count);
            self.count -= 1;
            
            // IMPORTANT: We must tell Tokio to poll us again!
            // Without this, Tokio would never come back to us.
            cx.waker().wake_by_ref();
            
            Poll::Pending
        }
    }
}

let rt = tokio::runtime::Runtime::new().unwrap();
rt.block_on(async {
    println!("  Starting countdown...");
    let result = Countdown { count: 3 }.await;  // .await calls poll() repeatedly!
    println!("  Result: {}", result);
});

### What Just Happened?

You just saw the **raw mechanics** of how a Future works:

1. We created a `Countdown { count: 3 }` and `.await`-ed it
2. `.await` told Tokio to `poll()` our Countdown
3. **First poll:** count=3, not zero, so we print, decrease to 2, call `wake()`, return `Pending`
4. **Second poll:** count=2, same thing, decrease to 1, return `Pending`
5. **Third poll:** count=1, decrease to 0, return `Pending`
6. **Fourth poll:** count=0, return `Ready("Liftoff!")`
7. `.await` gets the "Liftoff!" string and we print it

The key line is `cx.waker().wake_by_ref()`. This tells Tokio: **"I returned Pending, but immediately poll me again."** Without calling the waker, Tokio would never come back.

In real code, you wouldn't call `wake()` immediately — you'd tell the OS "wake me when data arrives" and the OS would call `wake()` later. But this shows the mechanism.

---

### Cell 8: A More Realistic Custom Future — Delayed Value

**What to watch for:** This Future waits until a specific time has passed, then returns a value. It uses the Waker properly — registering to be woken up later, not immediately.

In [ ]:
use std::time::Instant as StdInstant;
use std::time::Duration as StdDuration;

/// A Future that waits for a duration, then returns a value.
/// This is a simplified version of what tokio::time::sleep does.
struct DelayedValue {
    value: String,
    deadline: StdInstant,
    done: bool,
}

impl DelayedValue {
    fn new(value: &str, delay: StdDuration) -> Self {
        DelayedValue {
            value: value.to_string(),
            deadline: StdInstant::now() + delay,
            done: false,
        }
    }
}

impl Future for DelayedValue {
    type Output = String;
    
    fn poll(mut self: Pin<&mut Self>, cx: &mut Context<'_>) -> Poll<String> {
        if StdInstant::now() >= self.deadline {
            println!("    poll() -> Ready! (time's up)");
            Poll::Ready(self.value.clone())
        } else {
            println!("    poll() -> Pending (still waiting...)");
            
            // Schedule a wake-up for later
            // In real Tokio code, this would register with the timer system.
            // For this demo, we just immediately request re-poll.
            let waker = cx.waker().clone();
            let deadline = self.deadline;
            std::thread::spawn(move || {
                // Sleep in a background thread until deadline
                let remaining = deadline.saturating_duration_since(StdInstant::now());
                std::thread::sleep(remaining);
                waker.wake();  // Tell Tokio: "time's up, poll again!"
            });
            
            Poll::Pending
        }
    }
}

let rt = tokio::runtime::Runtime::new().unwrap();
rt.block_on(async {
    println!("  Waiting for delayed value...");
    let start = StdInstant::now();
    let value = DelayedValue::new("Hello from the future!", StdDuration::from_secs(2)).await;
    println!("  Got: '{}' after {:?}", value, start.elapsed());
});

### What Just Happened?

This is closer to how real async I/O works:

1. We created a `DelayedValue` that should produce a value after 2 seconds
2. **First poll:** Deadline hasn't passed, so:
   - We clone the Waker
   - We spawn a background thread that sleeps until the deadline
   - We return `Pending` — Tokio goes to do other things
3. **2 seconds pass...** The background thread wakes up and calls `waker.wake()`
4. Tokio hears the wake-up call and polls our Future again
5. **Second poll:** Deadline has passed! Return `Ready("Hello from the future!")`

**This is exactly the pattern:** Future asks something to notify it later (via Waker), returns Pending, and gets polled again when the notification arrives.

In real Tokio, the reactor handles this (using OS-level timers), not a background thread. But the principle is identical.

---

# Chapter 7: What The Compiler Does — State Machines

---

When you write an `async fn`, the Rust compiler transforms it into a **state machine**. This is one of the most important things to understand.

## Example

When you write:
```rust
async fn fetch_and_process() -> String {
    let raw = fetch_data().await;        // <-- await point 1
    let processed = process(raw).await;  // <-- await point 2
    format!("Result: {}", processed)
}
```

The compiler turns this into something like:
```rust
enum FetchAndProcessState {
    // Haven't started yet
    Start,
    // We called fetch_data(), waiting for it to complete
    WaitingForFetch { fetch_future: FetchDataFuture },
    // fetch_data() returned, we called process(), waiting for it
    WaitingForProcess { raw: String, process_future: ProcessFuture },
    // We're done
    Done,
}
```

Each `.await` creates a new state. The Future stores:
- Which state it's in
- All the local variables it needs for later states
- The inner Future it's currently waiting on

When Tokio calls `poll()`:
- **State = Start:** Call `fetch_data()`, save its future, move to `WaitingForFetch`
- **State = WaitingForFetch:** Poll the fetch future. If `Pending`, return `Pending`. If `Ready(raw)`, save `raw`, call `process(raw)`, move to `WaitingForProcess`
- **State = WaitingForProcess:** Poll the process future. If `Pending`, return `Pending`. If `Ready(processed)`, return `Ready(format!(...))`

## Why This Matters

1. **Zero cost:** No heap allocation per task (unlike Go goroutines or JS promises). The state machine is a fixed-size struct.
2. **No garbage collector:** Rust knows exactly when to drop the state machine.
3. **Predictable performance:** You can reason about exactly what memory each async task uses.

### Cell 9: Simulating the State Machine By Hand

**What to watch for:** We'll manually implement the state machine that the compiler would generate for an async function with two await points. This is purely educational — you'd never write this by hand.

In [ ]:
/// This is what the compiler creates when you write:
///
///   async fn two_step_job() -> String {
///       let a = step_one().await;
///       let b = step_two(a).await;
///       format!("{} then {}", a, b)
///   }

// The states our "async function" can be in
enum TwoStepState {
    NotStarted,
    WaitingStep1 { polls_left: u32 },
    WaitingStep2 { step1_result: String, polls_left: u32 },
    Completed,
}

struct TwoStepJob {
    state: TwoStepState,
}

impl Future for TwoStepJob {
    type Output = String;
    
    fn poll(mut self: Pin<&mut Self>, cx: &mut Context<'_>) -> Poll<String> {
        loop {
            match &mut self.state {
                TwoStepState::NotStarted => {
                    println!("    [State: NotStarted] -> Starting step 1...");
                    self.state = TwoStepState::WaitingStep1 { polls_left: 2 };
                    // Continue the loop to immediately poll the next state
                }
                
                TwoStepState::WaitingStep1 { polls_left } => {
                    if *polls_left == 0 {
                        println!("    [State: WaitingStep1] -> Step 1 complete!");
                        let result = "step1-data".to_string();
                        self.state = TwoStepState::WaitingStep2 {
                            step1_result: result,
                            polls_left: 2,
                        };
                        // Continue loop to start step 2
                    } else {
                        println!("    [State: WaitingStep1] -> Pending ({} polls left)", polls_left);
                        *polls_left -= 1;
                        cx.waker().wake_by_ref();
                        return Poll::Pending;
                    }
                }
                
                TwoStepState::WaitingStep2 { step1_result, polls_left } => {
                    if *polls_left == 0 {
                        println!("    [State: WaitingStep2] -> Step 2 complete!");
                        let result = format!("{} -> step2-data", step1_result);
                        self.state = TwoStepState::Completed;
                        return Poll::Ready(result);
                    } else {
                        println!("    [State: WaitingStep2] -> Pending ({} polls left)", polls_left);
                        *polls_left -= 1;
                        cx.waker().wake_by_ref();
                        return Poll::Pending;
                    }
                }
                
                TwoStepState::Completed => {
                    panic!("Polled after completion!");
                }
            }
        }
    }
}

let rt = tokio::runtime::Runtime::new().unwrap();
rt.block_on(async {
    println!("  Running our hand-made state machine...");
    let job = TwoStepJob { state: TwoStepState::NotStarted };
    let result = job.await;
    println!("  Final result: {}", result);
});

### What Just Happened?

You just manually built what the Rust compiler builds automatically when you write `async fn`! You can see:

1. The state machine transitions: `NotStarted -> WaitingStep1 -> WaitingStep2 -> Completed`
2. Each `.await` point = a place where the state machine can return `Pending` and pause
3. Data from earlier steps (`step1_result`) is stored in the state machine struct
4. When polled again, it picks up exactly where it left off

**The compiler does this transformation for you.** Every time you write `async fn`, Rust creates one of these state machine enums automatically. That's why async in Rust is "zero-cost" — it's just a struct with match arms, no heap allocation needed.

---

# Chapter 8: `select!` — Racing Futures Against Each Other

---

Sometimes you want to wait for **the first one to finish**, not all of them.

Real-world examples:
- Send a request to 3 mirror servers, use whichever responds first
- Wait for user input OR a timeout — whichever happens first
- Listen for a shutdown signal while also doing work

`tokio::select!` does this. It runs multiple futures and **returns as soon as ONE completes**, cancelling the rest.

### Cell 10: Racing With `select!`

**What to watch for:** We race a "fast server" (1 second) against a "slow server" (5 seconds). `select!` picks the winner and drops the loser.

In [ ]:
async fn fast_server() -> String {
    sleep(Duration::from_secs(1)).await;
    "Response from Fast Server".to_string()
}

async fn slow_server() -> String {
    sleep(Duration::from_secs(5)).await;
    "Response from Slow Server".to_string()
}

let rt = tokio::runtime::Runtime::new().unwrap();
rt.block_on(async {
    let start = Instant::now();
    
    // Race the two servers!
    tokio::select! {
        response = fast_server() => {
            println!("  Winner: {}", response);
        }
        response = slow_server() => {
            println!("  Winner: {}", response);
        }
    }
    
    println!("  Time: {:?}", start.elapsed());  // ~1 second, not 5!
});

### What Just Happened?

`select!` started both `fast_server()` and `slow_server()` at the same time. After 1 second, `fast_server` completed. `select!` immediately:
1. Returned the fast server's response
2. **Cancelled** the slow server (dropped its Future)

Total time: ~1 second. The slow server was abandoned because we didn't need it anymore.

---

### Cell 11: Timeout Pattern With `select!`

**What to watch for:** This is a very common pattern — "do this task, but give up if it takes too long." We'll use `select!` to implement a timeout.

In [ ]:
async fn slow_database_query() -> String {
    sleep(Duration::from_secs(10)).await;  // This takes 10 seconds!
    "query result".to_string()
}

let rt = tokio::runtime::Runtime::new().unwrap();
rt.block_on(async {
    println!("  Querying database with 2-second timeout...");
    let start = Instant::now();
    
    tokio::select! {
        result = slow_database_query() => {
            println!("  Got result: {}", result);
        }
        _ = sleep(Duration::from_secs(2)) => {
            println!("  TIMEOUT! Database took too long. Gave up after 2 seconds.");
        }
    }
    
    println!("  Time: {:?}", start.elapsed());
    
    // Tokio also has a built-in helper for this:
    println!("\n  --- Using tokio::time::timeout instead ---");
    let start = Instant::now();
    match tokio::time::timeout(Duration::from_secs(2), slow_database_query()).await {
        Ok(result) => println!("  Got result: {}", result),
        Err(_) => println!("  TIMEOUT! Gave up after 2 seconds."),
    }
    println!("  Time: {:?}", start.elapsed());
});

### What Just Happened?

The database query takes 10 seconds, but we only waited 2 seconds. Two ways to do it:

1. **`select!`** — race the query against a sleep timer. Whichever finishes first wins. The sleep won (2s < 10s), so we printed the timeout message.

2. **`tokio::time::timeout`** — a helper that does exactly this pattern. It returns `Ok(result)` if the future finishes in time, or `Err` if the timeout expired.

Both cancelled the slow database query — its Future was dropped, and no resources were wasted waiting for the remaining 8 seconds.

---

# Chapter 9: Channels — Sending Data Between Tasks

---

When you have multiple tasks running, they often need to **communicate**. One task produces data, another task consumes it.

Tokio provides **channels** for this. Think of a channel like a pipe:
- One end is the **sender** (puts data in)
- Other end is the **receiver** (takes data out)

## Types of Channels

| Channel | Description | When to use |
|---------|-------------|-------------|
| `mpsc` | Multiple senders, one receiver | Most common. Many producers, one consumer. |
| `oneshot` | One sender, one receiver, one message | When you need to send exactly one result back. |
| `broadcast` | One sender, multiple receivers | When many tasks need to hear the same message. |
| `watch` | One sender, multiple receivers, latest value only | For config updates or status changes. |

### Cell 12: `mpsc` Channel — Producer/Consumer Pattern

**What to watch for:** We'll spawn 3 "worker" tasks that produce results, and collect all results in the main task through a channel. This is very common in real programs.

In [ ]:
use tokio::sync::mpsc;

let rt = tokio::runtime::Runtime::new().unwrap();
rt.block_on(async {
    // Create a channel with buffer size 10
    // tx = sender (transmitter), rx = receiver
    let (tx, mut rx) = mpsc::channel::<String>(10);
    
    // Spawn 3 worker tasks, each gets a clone of the sender
    for i in 1..=3 {
        let tx = tx.clone();  // Each task gets its own sender
        tokio::spawn(async move {
            // Simulate doing some work
            sleep(Duration::from_millis(i * 500)).await;
            let message = format!("Worker {} finished after {}ms", i, i * 500);
            println!("  [Worker {}] Sending result...", i);
            tx.send(message).await.unwrap();
        });
    }
    
    // IMPORTANT: Drop the original sender!
    // The channel closes when ALL senders are dropped.
    // If we keep this one alive, rx.recv() will wait forever.
    drop(tx);
    
    // Collect results as they arrive
    println!("  [Main] Waiting for workers...\n");
    while let Some(message) = rx.recv().await {
        println!("  [Main] Received: {}", message);
    }
    
    println!("\n  [Main] All workers done! Channel closed.");
});

### What Just Happened?

1. We created a channel (`tx` for sending, `rx` for receiving)
2. We spawned 3 worker tasks. Each one got a **clone** of the sender (`tx.clone()`)
3. Workers do their "work" (sleep), then send their result through the channel
4. The main task receives results one by one with `rx.recv().await`
5. When all senders are dropped (workers finished + we dropped the original `tx`), `rx.recv()` returns `None` and the loop ends

**Why `drop(tx)`?** We cloned `tx` for each worker, but the original `tx` still exists. If we don't drop it, the channel never closes, and `rx.recv().await` waits forever. Always drop the original sender after cloning.

---

### Cell 13: `oneshot` Channel — Getting One Result Back

**What to watch for:** `oneshot` is perfect when you spawn a task and want to get ONE result back from it. Like sending someone on an errand and waiting for them to come back with the answer.

In [ ]:
use tokio::sync::oneshot;

let rt = tokio::runtime::Runtime::new().unwrap();
rt.block_on(async {
    // Create a oneshot channel
    let (tx, rx) = oneshot::channel::<String>();
    
    // Spawn a task to "compute" something
    tokio::spawn(async move {
        println!("  [Task] Computing the answer to life...");
        sleep(Duration::from_secs(1)).await;
        let answer = "42".to_string();
        println!("  [Task] Sending answer back!");
        tx.send(answer).unwrap();  // Send the one result
    });
    
    // Wait for the answer
    println!("  [Main] Waiting for the answer...");
    let answer = rx.await.unwrap();
    println!("  [Main] The answer is: {}", answer);
});

### What Just Happened?

Simple and clean:
1. Create a `oneshot` channel (one sender, one receiver, one message)
2. Give the sender to a spawned task
3. The task does work and sends back exactly one result
4. The main task awaits on the receiver to get the result

Use `oneshot` when you need to get a single response from a background task. Use `mpsc` when a task sends multiple messages.

---

# Chapter 10: Real-World Example — A Simple TCP Echo Server

---

Now let's put it all together with something real: a **TCP echo server**. This server:
1. Listens for connections on a port
2. When a client connects, reads whatever they send
3. Sends it back (echo)
4. Handles **many clients at the same time** using `tokio::spawn`

This is directly relevant to your reverse proxy project — a reverse proxy is basically this but forwarding data instead of echoing it.

### Cell 14: TCP Echo Server

**What to watch for:** 
- `TcpListener::bind` — starts listening on a port
- `listener.accept().await` — waits for a client to connect (without blocking!)
- `tokio::spawn` — each client gets its own task, so thousands of clients can connect simultaneously
- `socket.read` / `socket.write_all` — async reading and writing

**Note:** This cell starts a server that runs for 5 seconds. You can test it by opening another terminal and running: `echo "hello" | nc localhost 8080`

In [ ]:
use tokio::net::TcpListener;
use tokio::io::{AsyncReadExt, AsyncWriteExt};

let rt = tokio::runtime::Runtime::new().unwrap();
rt.block_on(async {
    // Step 1: Bind to a port (start listening)
    let listener = TcpListener::bind("127.0.0.1:8080").await.unwrap();
    println!("  Echo server listening on 127.0.0.1:8080");
    println!("  (will auto-stop after 5 seconds)\n");
    
    // We'll auto-stop after 5 seconds for this demo
    tokio::select! {
        _ = async {
            loop {
                // Step 2: Accept a new connection
                // This .await pauses until someone connects.
                // While waiting, other tasks (other clients) keep running!
                let (mut socket, addr) = listener.accept().await.unwrap();
                println!("  New client connected: {}", addr);
                
                // Step 3: Spawn a task for this client
                // This means we immediately go back to accepting more clients!
                tokio::spawn(async move {
                    let mut buf = [0u8; 1024];
                    
                    loop {
                        // Step 4: Read data from the client
                        let n = match socket.read(&mut buf).await {
                            Ok(0) => {
                                // 0 bytes = client disconnected
                                println!("  Client {} disconnected", addr);
                                return;
                            }
                            Ok(n) => n,
                            Err(e) => {
                                println!("  Error reading from {}: {}", addr, e);
                                return;
                            }
                        };
                        
                        let received = String::from_utf8_lossy(&buf[..n]);
                        println!("  Received from {}: {}", addr, received.trim());
                        
                        // Step 5: Echo it back
                        if socket.write_all(&buf[..n]).await.is_err() {
                            println!("  Error writing to {}", addr);
                            return;
                        }
                    }
                });
            }
        } => {}
        _ = sleep(Duration::from_secs(5)) => {
            println!("  Server shutting down after 5 seconds.");
        }
    }
});

### What Just Happened?

This is a **complete, production-style async server** (simplified). Let's trace the flow:

1. **Bind:** The server starts listening on port 8080
2. **Accept loop:** `listener.accept().await` waits for someone to connect. The `.await` means other tasks can run while we wait — we're not blocking!
3. **Spawn per client:** When a client connects, we `tokio::spawn` a new task for that client. This is key — the accept loop immediately goes back to waiting for the next client. We can handle thousands of simultaneous connections.
4. **Read/Write loop:** Each client task reads data, prints it, and writes it back (echo). All the reads and writes are `.await`-ed, so they don't block other clients.
5. **`select!` for shutdown:** The whole server races against a 5-second timer. When the timer wins, the server stops.

**This is the pattern your reverse proxy will use** — except instead of echoing data back, it will forward data to another server.

---

# Chapter 11: Common Mistakes That Will Bite You

---

These are the mistakes every Rust async beginner makes. Read carefully!

### Cell 15: Mistake #1 — Blocking the Runtime

**What to watch for:** We'll show the difference between blocking sleep (`std::thread::sleep`) and async sleep (`tokio::time::sleep`). The blocking version **freezes** other tasks.

In [ ]:
let rt = tokio::runtime::Builder::new_current_thread()
    .enable_all()
    .build()
    .unwrap();

rt.block_on(async {
    // === BAD: Using blocking sleep ===
    println!("=== BAD: std::thread::sleep (blocking) ===");
    let start = Instant::now();
    
    let a = tokio::spawn(async {
        println!("  Task A starting");
        std::thread::sleep(std::time::Duration::from_secs(1));  // BLOCKS the thread!
        println!("  Task A done at {:?}", Instant::now());
    });
    let b = tokio::spawn(async {
        println!("  Task B starting");
        std::thread::sleep(std::time::Duration::from_secs(1));  // BLOCKS the thread!
        println!("  Task B done at {:?}", Instant::now());
    });
    a.await.unwrap();
    b.await.unwrap();
    println!("  Total: {:?}\n", start.elapsed());  // ~2 seconds! (not 1)
    
    // === GOOD: Using async sleep ===
    println!("=== GOOD: tokio::time::sleep (non-blocking) ===");
    let start = Instant::now();
    
    let a = tokio::spawn(async {
        println!("  Task A starting");
        sleep(Duration::from_secs(1)).await;  // Yields to runtime!
        println!("  Task A done at {:?}", Instant::now());
    });
    let b = tokio::spawn(async {
        println!("  Task B starting");
        sleep(Duration::from_secs(1)).await;  // Yields to runtime!
        println!("  Task B done at {:?}", Instant::now());
    });
    a.await.unwrap();
    b.await.unwrap();
    println!("  Total: {:?}", start.elapsed());  // ~1 second!
});

### What Just Happened?

With the **single-threaded runtime**:

- **Blocking sleep:** Task A blocks the only thread for 1 second. Task B can't even start until Task A's sleep finishes. Total: **~2 seconds**.
- **Async sleep:** Task A hits `.await` and yields. Task B starts immediately. Both sleep at the same time. Total: **~1 second**.

**The rule:** Never use these in async code:
- `std::thread::sleep()` → use `tokio::time::sleep()` instead
- `std::fs::read()` → use `tokio::fs::read()` instead
- Any function that blocks the thread

If you MUST call blocking code (e.g., a library that doesn't have async support), wrap it in `tokio::task::spawn_blocking()`.

---

### Cell 16: Mistake #2 — Holding a Lock Across `.await`

**What to watch for:** If you lock a `std::sync::Mutex` and then `.await` while holding the lock, you can deadlock the whole program. Other tasks can't get the lock because you're holding it, but you're paused waiting for something else.

In [ ]:
use std::sync::Arc;

let rt = tokio::runtime::Runtime::new().unwrap();
rt.block_on(async {

    // === BAD: Holding std::sync::Mutex across .await ===
    // DON'T DO THIS (commented out because it can deadlock):
    //
    // let data = Arc::new(std::sync::Mutex::new(vec![]));
    // let lock = data.lock().unwrap();  // Lock acquired
    // sleep(Duration::from_secs(1)).await;  // Paused while holding lock!
    // lock.push(1);  // Other tasks can't access `data` this whole time!
    // drop(lock);
    
    // === GOOD: Use tokio::sync::Mutex for async code ===
    println!("=== GOOD: tokio::sync::Mutex ===");
    let data = Arc::new(tokio::sync::Mutex::new(Vec::new()));
    
    let data1 = data.clone();
    let t1 = tokio::spawn(async move {
        let mut lock = data1.lock().await;  // Async lock - yields while waiting!
        println!("  Task 1: got the lock, working...");
        sleep(Duration::from_millis(500)).await;
        lock.push(1);
        println!("  Task 1: done, releasing lock");
    });
    
    let data2 = data.clone();
    let t2 = tokio::spawn(async move {
        let mut lock = data2.lock().await;  // Will wait until Task 1 releases
        println!("  Task 2: got the lock, working...");
        sleep(Duration::from_millis(500)).await;
        lock.push(2);
        println!("  Task 2: done, releasing lock");
    });
    
    t1.await.unwrap();
    t2.await.unwrap();
    
    let result = data.lock().await;
    println!("  Final data: {:?}", *result);
});

### What Just Happened?

**The problem with `std::sync::Mutex` in async:**
- You lock it → you `.await` something → your task pauses → the lock is STILL held → other tasks waiting for the lock are blocked → potential deadlock

**The solution: `tokio::sync::Mutex`**
- Its `.lock()` is async — if the lock is taken, your task **yields** (lets other tasks run) instead of blocking the thread
- It's safe to hold across `.await` points

**Quick rule:**
- If you hold the lock briefly and never `.await` while holding it → `std::sync::Mutex` is fine (and faster)
- If you need to `.await` while holding the lock → use `tokio::sync::Mutex`

---

# Chapter 12: `Pin` — Why Async Needs It

---

This is the concept that confuses most people, but it's simpler than it looks.

## The Problem

Remember how `async fn` becomes a state machine struct? That struct stores local variables. Sometimes, a local variable **points to another local variable** in the same struct.

```rust
async fn example() {
    let data = vec![1, 2, 3];      // stored in the state machine
    let reference = &data;          // points to `data` inside the same struct!
    some_async_thing().await;       // state machine might be moved here!
    println!("{:?}", reference);    // if the struct moved, this pointer is WRONG!
}
```

If Tokio moves this state machine to a different memory location (e.g., moving it to a different thread), `reference` would point to the **old** location of `data`. That's a dangling pointer — undefined behavior!

## The Solution: `Pin`

`Pin<&mut T>` is a wrapper that says: **"This value will NOT be moved in memory."**

When Tokio calls `poll()`, the signature is:
```rust
fn poll(self: Pin<&mut Self>, cx: &mut Context) -> Poll<Output>
```

The `Pin<&mut Self>` guarantees: "This Future struct is pinned. It won't move. Internal references are safe."

## Do I Need To Worry About Pin?

**Usually no.** When you use `async fn` and `.await`, the compiler handles Pin automatically. You only need to think about Pin when:
1. You implement `Future` by hand (like we did in Cell 7)
2. You use `tokio::pin!()` macro to pin a local future
3. You store futures in collections

For now, just understand **why** it exists: to prevent the async state machine from being moved in memory when it has internal references.

---

# Chapter 13: Putting It All Together — A Mini Project

---

Let's build something that uses everything we've learned: a **concurrent URL health checker**.

It will:
1. Take a list of "URLs" (simulated)
2. Check all of them **concurrently** using `tokio::spawn`
3. Use a **channel** to send results back
4. Use a **timeout** so slow URLs don't hold us up
5. Print a summary at the end

### Cell 17: Concurrent URL Health Checker

**What to watch for:** This uses `spawn`, `mpsc` channels, `timeout`, and `select!` — all the tools from this guide combined into one working program.

In [ ]:
use tokio::sync::mpsc;
use tokio::time::{timeout, Duration, Instant, sleep};

// Simulated "URL check" — in real code this would be an HTTP request
async fn check_url(url: &str, latency_ms: u64, should_fail: bool) -> Result<u16, String> {
    sleep(Duration::from_millis(latency_ms)).await;
    if should_fail {
        Err(format!("Connection refused: {}", url))
    } else {
        Ok(200)  // HTTP 200 OK
    }
}

#[derive(Debug)]
struct HealthResult {
    url: String,
    status: String,
    duration_ms: u128,
}

let rt = tokio::runtime::Runtime::new().unwrap();
rt.block_on(async {
    println!("  === URL Health Checker ===");
    println!("  Checking 5 URLs concurrently...\n");
    
    let overall_start = Instant::now();
    
    // URLs to check: (name, simulated latency, should_fail)
    let urls = vec![
        ("https://google.com",     200,  false),
        ("https://github.com",     500,  false),
        ("https://slow-site.com",  3000, false),  // This will timeout!
        ("https://broken.com",     100,  true),   // This will fail!
        ("https://rust-lang.org",  300,  false),
    ];
    
    // Channel to collect results
    let (tx, mut rx) = mpsc::channel::<HealthResult>(10);
    
    // Spawn a health check task for EACH url (all run concurrently)
    for (url, latency, should_fail) in urls {
        let tx = tx.clone();
        let url_string = url.to_string();
        
        tokio::spawn(async move {
            let start = Instant::now();
            
            // 1-second timeout for each check
            let status = match timeout(
                Duration::from_secs(1),
                check_url(&url_string, latency, should_fail)
            ).await {
                Ok(Ok(code)) => format!("OK ({})", code),
                Ok(Err(e))   => format!("FAIL ({})", e),
                Err(_)       => "TIMEOUT (>1s)".to_string(),
            };
            
            let result = HealthResult {
                url: url_string,
                status,
                duration_ms: start.elapsed().as_millis(),
            };
            
            tx.send(result).await.ok();
        });
    }
    
    // Drop original sender so channel closes when all tasks complete
    drop(tx);
    
    // Collect and display results
    let mut results = Vec::new();
    while let Some(result) = rx.recv().await {
        results.push(result);
    }
    
    // Sort by URL for consistent display
    results.sort_by(|a, b| a.url.cmp(&b.url));
    
    println!("  {:<30} {:<25} {:>8}", "URL", "STATUS", "TIME");
    println!("  {}", "-".repeat(65));
    for r in &results {
        println!("  {:<30} {:<25} {:>6}ms", r.url, r.status, r.duration_ms);
    }
    
    let ok_count = results.iter().filter(|r| r.status.starts_with("OK")).count();
    println!("\n  Summary: {}/{} healthy", ok_count, results.len());
    println!("  Total time: {:?} (all checked concurrently!)", overall_start.elapsed());
});

### What Just Happened?

We built a real tool that combines everything:

| Concept | Where it was used |
|---------|------------------|
| `async fn` | `check_url()` — the simulated health check |
| `tokio::spawn` | Each URL gets its own concurrent task |
| `.await` | Every I/O operation yields to the runtime |
| `mpsc` channel | Workers send results back to the main task |
| `timeout` | Slow URLs (>1s) are cancelled |
| `move` closures | Each task owns its data |

**Key insight:** All 5 URLs were checked **at the same time**. The total time was ~1 second (the timeout duration), not 4.1 seconds (200+500+3000+100+300ms). That's the power of async.

---

# Chapter 14: Quick Reference — Cheat Sheet

---

## Essentials

```rust
// Make a function async
async fn do_thing() -> Result<String, Error> { ... }

// Call an async function (must be inside another async fn)
let result = do_thing().await;

// Start the Tokio runtime
#[tokio::main]
async fn main() { ... }

// Spawn a background task
let handle = tokio::spawn(async move { ... });
let result = handle.await.unwrap();
```

## Running Multiple Things

```rust
// Run all, wait for all (concurrent)
let (a, b, c) = tokio::join!(future_a, future_b, future_c);

// Run all, return the first to finish
tokio::select! {
    val = future_a => { /* a finished first */ }
    val = future_b => { /* b finished first */ }
}

// Add a timeout
match tokio::time::timeout(Duration::from_secs(5), future).await {
    Ok(result) => { /* finished in time */ }
    Err(_) => { /* timed out */ }
}
```

## Channels

```rust
// Multiple messages
let (tx, mut rx) = tokio::sync::mpsc::channel(buffer_size);
tx.send(value).await;
let val = rx.recv().await;

// Single message
let (tx, rx) = tokio::sync::oneshot::channel();
tx.send(value).unwrap();
let val = rx.await.unwrap();
```

## Networking

```rust
use tokio::net::{TcpListener, TcpStream};
use tokio::io::{AsyncReadExt, AsyncWriteExt};

// Server
let listener = TcpListener::bind("0.0.0.0:8080").await?;
let (socket, addr) = listener.accept().await?;

// Client
let mut stream = TcpStream::connect("127.0.0.1:8080").await?;
stream.write_all(b"hello").await?;
let n = stream.read(&mut buf).await?;
```

## Rules of Thumb

| Do | Don't |
|----|---------|
| `tokio::time::sleep()` | `std::thread::sleep()` |
| `tokio::fs::read()` | `std::fs::read()` |
| `tokio::sync::Mutex` (if holding across .await) | `std::sync::Mutex` across .await |
| `tokio::spawn(async move { ... })` | borrowing data in spawned tasks |
| Always `.await` your futures | Calling async fn without `.await` |

# What's Next?

---

Now that you understand async and Tokio, here's the path forward toward your reverse proxy project:

1. **TCP streams in depth** — reading/writing bytes, handling partial reads, buffering
2. **Tokio's `io::copy`** — efficiently forwarding bytes between two sockets (the core of a proxy)
3. **TLS with `tokio-rustls`** — adding encryption to your connections
4. **HTTP parsing** — understanding what bytes are flowing through your proxy
5. **Graceful shutdown** — cleaning up when the server needs to stop

Each of these builds directly on what you learned here. The patterns are always the same: spawn tasks, await I/O, use channels to communicate.

---

*Happy coding!*